# Giving an LLM tools: a buy/hold/sell recommendation

A language model on its own can only tell you what it absorbed during training. Ask it
about a company and it will answer confidently, in the voice of an analyst, using
numbers it half-remembers from a world that has since moved on.

An **agent** is the fix, and it is a small idea: let the model call functions. We give it
a loop. Each time round, the model either answers or asks to run one of our functions.
We run it, hand back the result, and ask again.

Today we build that up in four stages, and at each stage we look at what the answer is
still missing:

| Stage | What the model can reach | What is still wrong |
|---|---|---|
| 1 | nothing | invented figures |
| 2 | a calculator | precise arithmetic on invented figures |
| 3 | web search | current context, but as prose |
| 4 | financial data | figures you can check |

Run the cells in order. Your two jobs at the end: rewrite the prompt, and add a tool.

In [ ]:
# Colab starts empty every session, so install what we need.
# The versions are pinned deliberately: these are the ones this notebook was
# built and tested against, and yfinance in particular changes often enough
# that an unpinned install could behave differently from one week to the next.
!pip install -q openai==3.42.0 yfinance==1.7.0 ddgs==9.16.1

In [ ]:
import getpass
import importlib
import os
from pathlib import Path

DEFAULT_BASE_URL = "https://api.openai.com/v1"
DEFAULT_MODEL = "gpt-4o-mini"


class CredentialError(Exception):
    """Raised when credentials are present but the provider rejects them."""

    @classmethod
    def unauthorized(cls) -> "CredentialError":
        return cls(
            "The API key was rejected. Check that you pasted the whole key, then "
            "re-enter it: open the key icon in the left sidebar to fix the Colab "
            "secret, or use Runtime -> Restart session and run the cells again."
        )

    @classmethod
    def unknown_model(cls, model: str | None = None) -> "CredentialError":
        return cls(
            f"The provider does not recognise the model {model!r}. Model names get "
            "retired: set a MODEL secret (key icon in the left sidebar) to one your "
            "account has, then use Runtime -> Restart session."
        )

    @classmethod
    def unreachable(cls, base_url: str | None = None) -> "CredentialError":
        return cls(
            f"Could not reach the model provider at {base_url!r}. Check the BASE_URL "
            "secret. Note that an address on a private network -- anything like "
            "192.168.x.x, 10.x.x.x or localhost -- is not reachable from Colab, "
            "because this notebook runs on a Google server rather than on your machine."
        )

    @classmethod
    def rate_limited(cls) -> "CredentialError":
        return cls(
            "The provider is rate-limiting this key -- usually because everyone in "
            "the room is calling it at once. Wait a few seconds and run the cell again."
        )


def _from_colab(name: str) -> str | None:
    """Read a Colab secret, or None when not on Colab / not set / not granted."""
    try:
        userdata = importlib.import_module("google.colab.userdata")
        value = userdata.get(name)
    except Exception:
        return None
    return value or None


def _load_dotenv(path: str = ".env") -> None:
    """Copy `KEY=value` lines from a .env file into the environment, if one exists.

    Colab has no .env file, so this does nothing there -- it is what makes the same
    notebook run locally, where a .env is the usual place to keep these. Values
    already in the environment win, so an explicit export still overrides the file.
    """
    try:
        text = Path(path).read_text(encoding="utf-8")
    except OSError:
        return
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        os.environ.setdefault(key.strip(), value.strip().strip("\"'"))


def _lookup(name: str) -> str | None:
    """Colab secret first, then the environment (which includes any .env file)."""
    return _from_colab(name) or os.getenv(name) or None


def get_credentials() -> tuple[str, str, str]:
    """Return (base_url, api_key, model), asking for the key only if nothing has it.

    Each value is looked for in the same order: a Colab secret, then the
    environment (including a .env file if one is present), then the default
    above. Only the API key has no default, so only the API key can prompt.
    """
    _load_dotenv()
    base_url = _lookup("BASE_URL") or DEFAULT_BASE_URL
    model = _lookup("MODEL") or DEFAULT_MODEL
    api_key = _lookup("OPENAI_API_KEY")
    if not api_key:
        api_key = getpass.getpass("Paste the API key for this session: ").strip()
    return base_url, api_key, model


BASE_URL, API_KEY, MODEL = get_credentials()
print(f"Using model {MODEL} at {BASE_URL}")

In [ ]:
# Short Code for Slides
from openai import OpenAI

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

messages = [{"role": "user", "content": "Hallo, wie geht es dir?"}]
messages=[
    {"role": "system", "content": "You are a stock analyst"},
    {"role": "user", "content": "Give your buy/hold/sell recommendation for Roche."},
],

try:
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=messages
    )
    print(response.choices[0].message.content)
except Exception as exc:
    print(f"unexpected error: {exc}")


In [ ]:
COMPANY = "Alphabet Inc. Class A (GOOGL)"
TICKER = "GOOGL"

SYSTEM_PROMPT = (
    "You are a stock analyst and have to give a buy/hold/sell recommendation about a company."
)

def _user_message(company: str) -> str:
    """What we actually ask for. Kept apart from SYSTEM_PROMPT, which is fixed."""
    return (
        f"Give your buy/hold/sell recommendation for {company}. "
        "Back it with specific, current figures rather than ones you recall, "
        "and list the figures you relied on so they can be checked."
    )

In [ ]:
# Stage 1: the model alone. One call, no tools, no loop.
def ask_without_tools(client, model: str, company: str) -> str:
    """Stage 1: one call, no tools, no loop. Whatever comes back is all we get."""
    response = _create(
        client,
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": _user_message(company)},
        ],
    )
    return response.choices[0].message.content

print(ask_without_tools(client, MODEL, COMPANY))

## What just went wrong

Read the answer again, and this time check it.

**Find every number it gave you** -- the price, the P/E, the market cap -- and look them
up. Compare notes with the person next to you: you are probably holding different
numbers, and none of them are today's.

The model is not lying. It is doing the only thing it can: recalling. Its training data
stopped at some point in the past, and it has no way to know that, so it reports old
figures in the present tense and sounds exactly as certain either way.

Two things make this sharper for Alphabet specifically:

- **The 2022 split.** Alphabet split 20-for-1. A model reaching for a pre-split price is
  not a few percent out, it is out by a factor of twenty. Watch for a suspiciously large
  share price.
- **It reads well.** Alphabet is one of the most written-about companies on earth, so the
  *narrative* the model produces is genuinely competent. Only the figures are wrong. That
  combination -- sound reasoning over broken inputs -- is the dangerous one, and it is
  much harder to spot than obvious nonsense.

No amount of rewording the prompt fixes this. You cannot ask a model to know something it
was never told. It needs a way to go and look.

In [ ]:
import json
from dataclasses import dataclass


@dataclass
class AgentResult:
    answer: str
    model_calls: int
    tool_calls: list
    stopped_early: bool


def _invoke(call, registry: dict) -> str:
    """Run one tool call and return its result as text. Never raises.

    A tool that raised would end the conversation instead of informing it, so every
    failure here becomes a message the model can read and react to -- which is what
    lets a student's half-finished tool fail without killing the cell.
    """
    name = call.function.name
    function = registry.get(name)
    if function is None:
        return f"Error: unknown tool {name}"
    try:
        arguments = json.loads(call.function.arguments or "{}")
    except json.JSONDecodeError as exc:
        return f"Error: could not read the arguments for {name}: {exc}"
    try:
        return str(function(**arguments))
    except Exception as exc:
        return f"Error: {exc}"


def run_agent(
    client,
    model: str,
    company: str,
    schemas: list[dict],
    registry: dict,
    max_rounds: int = 10,
) -> AgentResult:
    """Stages 2-4: call the model, run any tool it asks for, repeat until it answers."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": _user_message(company)},
    ]
    model_calls = 0
    called: list[str] = []

    for _ in range(max_rounds):
        response = _create(
            client,
            model=model,
            messages=messages,
            tools=schemas or None,
        )
        model_calls += 1
        message = response.choices[0].message
        messages.append(message)

        if not message.tool_calls:
            return AgentResult(message.content, model_calls, called, stopped_early=False)

        for call in message.tool_calls:
            called.append(call.function.name)
            result = _invoke(call, registry)
            messages.append(
                {"role": "tool", "tool_call_id": call.id, "content": result}
            )

    # The model kept asking for tools and never settled. Stop rather than spend the
    # shared key on a loop that is not converging. The default of 10 comes from a
    # live run where a search-happy model used 6 rounds on stage 3 alone and was
    # cut off mid-answer.
    last = next(
        (m.content for m in reversed(messages) if getattr(m, "content", None)),
        None,
    )
    return AgentResult(
        last or f"Stopped after {max_rounds} rounds without a final answer.",
        model_calls,
        called,
        stopped_early=True,
    )

In [ ]:
import ast

_ALLOWED_NODES = (
    ast.Expression,
    ast.Constant,
    ast.UnaryOp,
    ast.BinOp,
    ast.USub,
    ast.UAdd,
    ast.Add,
    ast.Sub,
    ast.Mult,
    ast.Div,
    ast.FloorDiv,
    ast.Mod,
    ast.Pow,
)

_MAX_EXPONENT = 1000

_BINARY_OPS = {
    ast.Add: lambda a, b: a + b,
    ast.Sub: lambda a, b: a - b,
    ast.Mult: lambda a, b: a * b,
    ast.Div: lambda a, b: a / b,
    ast.FloorDiv: lambda a, b: a // b,
    ast.Mod: lambda a, b: a % b,
    ast.Pow: lambda a, b: a**b,
}


def _evaluate(node):
    """Walk the validated tree and compute its value."""
    if isinstance(node, ast.Expression):
        return _evaluate(node.body)
    if isinstance(node, ast.Constant):
        return node.value
    if isinstance(node, ast.UnaryOp):
        value = _evaluate(node.operand)
        return -value if isinstance(node.op, ast.USub) else +value
    if isinstance(node, ast.BinOp):
        left, right = _evaluate(node.left), _evaluate(node.right)
        if isinstance(node.op, ast.Pow) and right > _MAX_EXPONENT:
            raise ValueError(f"exponent above {_MAX_EXPONENT} is too large to compute")
        return _BINARY_OPS[type(node.op)](left, right)
    raise ValueError("unsupported expression")


def calculator(expression: str) -> str:
    """Evaluate an arithmetic expression, e.g. "1234.5 * 0.21"."""
    try:
        tree = ast.parse(expression, mode="eval")
    except SyntaxError:
        return f"Error: {expression!r} is not a valid arithmetic expression."

    for node in ast.walk(tree):
        if not isinstance(node, _ALLOWED_NODES):
            return (
                f"Error: {expression!r} contains something that is not arithmetic. "
                "Pass numbers and the operators + - * / // % ** only."
            )
        if isinstance(node, ast.Constant) and not isinstance(node.value, (int, float)):
            return f"Error: {node.value!r} is not a number."
        if isinstance(node, ast.Constant) and isinstance(node.value, bool):
            return f"Error: {node.value!r} is not a number."

    try:
        return str(_evaluate(tree))
    except ZeroDivisionError:
        return "Error: cannot divide by zero."
    except OverflowError:
        return "Error: that result is too large to compute."
    except ValueError as exc:
        return f"Error: {exc}."


CALCULATOR_SCHEMA = {
    "type": "function",
    "function": {
        "name": "calculator",
        "description": (
            "Evaluate one arithmetic expression and return the result. "
            "Use this for any ratio, growth rate, or multiple you need to compute "
            "rather than doing the arithmetic yourself. Pass numbers and the "
            "operators + - * / // % ** only -- no variable names, no function calls."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "The arithmetic to evaluate, e.g. '(180 - 150) / 150 * 100'",
                }
            },
            "required": ["expression"],
        },
    },
}

In [ ]:
# Stage 2: the same question, but now the model can do arithmetic.
STAGE_2_SCHEMAS = [CALCULATOR_SCHEMA]
STAGE_2_REGISTRY = {"calculator": calculator}

result = run_agent(client, MODEL, COMPANY, STAGE_2_SCHEMAS, STAGE_2_REGISTRY)
print(result.answer)
print(f"\n--- {result.model_calls} model calls, tools used: {result.tool_calls}")
if result.stopped_early:
    print("--- NOTE: hit the round limit, so the answer above is incomplete.")

In [ ]:
from ddgs import DDGS

# The offline snapshot, embedded because Colab has no copy of the repository.
# It is used only when DuckDuckGo throttles us -- which it will, with everyone
# searching at once.
CACHE = {
    "generated_at": "",
    "ticker": "GOOGL",
    "company": "Alphabet",
    "aliases": [
        "GOOGL",
        "GOOG",
        "Alphabet",
        "Google"
    ],
    "fundamentals": {},
    "search": [
        {
            "title": "PLACEHOLDER - run tools/make_snapshot.py to fetch real GOOGL coverage",
            "body": "This snapshot has not been generated yet. Until it is, the offline fallback returns this notice instead of real search results, so that nobody mistakes seeded text for genuine reporting on Alphabet.",
            "href": "https://example.invalid/placeholder"
        },
        {
            "title": "PLACEHOLDER - no cached news available",
            "body": "Regenerate before the lecture: uv run python tools/make_snapshot.py",
            "href": "https://example.invalid/placeholder"
        }
    ]
}


def load_cache():
    return CACHE


def _ddgs_text(query: str, max_results: int) -> list[dict]:
    """The live DuckDuckGo call. Separated out so tests can stand in for the network."""
    from ddgs import DDGS

    return list(DDGS().text(query, max_results=max_results))


def _format_results(results: list[dict]) -> str:
    lines = []
    for i, r in enumerate(results, start=1):
        lines.append(f"{i}. {r.get('title', '')}\n   {r.get('body', '')}\n   {r.get('href', '')}")
    return "\n".join(lines)


def web_search(query: str, max_results: int = 5) -> str:
    """Search the web and return the top results as text."""
    max_results = max(1, min(int(max_results or 5), 10))
    try:
        results = _ddgs_text(query, max_results)
    except Exception:
        results = []
    if results:
        return _format_results(results)

    # DuckDuckGo throttles when a whole classroom searches at once. That is an
    # expected outcome here, not an error: fall back to the cached snapshot and
    # say nothing, so a throttled student gets the same lesson as everyone else.
    #
    # But the snapshot is about ONE company. Serving its headlines for a query
    # about some other company would be inventing news, so the fallback only
    # applies when the query is actually about the company we cached.
    cache = load_cache()
    aliases = cache.get("aliases") or [cache.get("ticker", "")]
    if not any(alias and alias.lower() in query.lower() for alias in aliases):
        return (
            f"Error: the live search failed and there is no offline snapshot for "
            f"{query!r}. The cached snapshot only covers {cache.get('company') or cache.get('ticker')}."
        )
    cached = cache.get("search") or []
    if not cached:
        return (
            "Error: the live search failed and the offline snapshot holds no results. "
            "Ask whoever prepared this notebook to regenerate it."
        )
    return _format_results(cached[:max_results])


WEB_SEARCH_SCHEMA = {
    "type": "function",
    "function": {
        "name": "web_search",
        "description": (
            "Search the web and return the top results as text. Use this for current "
            "news, recent events, and analyst commentary -- anything that happened "
            "after your training data ends. Results are prose, not figures: use "
            "get_fundamentals when you need numbers you can compute with."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "What to search for, e.g. 'Alphabet Q3 earnings analyst reaction'",
                },
                "max_results": {
                    "type": "integer",
                    "description": "How many results to return. Defaults to 5.",
                },
            },
            "required": ["query"],
        },
    },
}

In [ ]:
# Stage 3: add web search. Note that run_agent has not changed -- the list grew.
STAGE_3_SCHEMAS = [CALCULATOR_SCHEMA, WEB_SEARCH_SCHEMA]
STAGE_3_REGISTRY = {"calculator": calculator, "web_search": web_search}

result = run_agent(client, MODEL, COMPANY, STAGE_3_SCHEMAS, STAGE_3_REGISTRY)
print(result.answer)
print(f"\n--- {result.model_calls} model calls, tools used: {result.tool_calls}")
if result.stopped_early:
    print("--- NOTE: hit the round limit, so the answer above is incomplete.")

In [ ]:
import yfinance

# Share classes are not interchangeable: GOOGL (Class A, voting) and GOOG (Class C,
# non-voting) trade at different prices. An agent that fetches whichever one it felt
# like has given you an answer about a different security than the one you asked
# about. The schema below tells the model so explicitly.

FUNDAMENTAL_FIELDS = [
    "currentPrice",
    "trailingPE",
    "forwardPE",
    "marketCap",
    "returnOnEquity",
    "profitMargins",
    "totalRevenue",
    "freeCashflow",
]


def _yf_info(ticker: str) -> dict:
    """The live Yahoo call. Separated out so tests can stand in for the network."""
    import yfinance

    return yfinance.Ticker(ticker).info


def get_fundamentals(ticker: str) -> str:
    """Look up the headline financials for a ticker, e.g. "GOOGL"."""
    try:
        info = _yf_info(ticker)
    except Exception:
        # yfinance reads an undocumented endpoint that breaks without warning. The
        # cached snapshot keeps the lecture running; the student sees no difference.
        #
        # The snapshot covers ONE ticker, though. Filling an AAPL heading with the
        # cached GOOGL figures would attribute real numbers to the wrong security --
        # and GOOG vs GOOGL is the very share-class trap this lecture teaches. So the
        # fallback applies only when the ticker asked for is the one we cached.
        cache = load_cache()
        if ticker.strip().upper() != str(cache.get("ticker", "")).strip().upper():
            return (
                f"Error: the live lookup for {ticker} failed and there is no offline "
                f"snapshot for it (the snapshot covers {cache.get('ticker')} only). "
                f"Do not substitute figures from another ticker."
            )
        info = cache.get("fundamentals") or {}
    lines = [f"{ticker} fundamentals:"]
    for field in FUNDAMENTAL_FIELDS:
        if info.get(field) is not None:
            lines.append(f"{field}: {info[field]}")
    if len(lines) == 1:
        # Yahoo failed and the snapshot was never generated. Say so rather than
        # hand back a bare heading: a silent empty result makes stage 4 look like
        # stage 1, and nobody in the room would know why.
        lines.append(
            "no data available -- the live lookup failed and the offline snapshot is "
            "empty. Ask whoever prepared this notebook to regenerate it."
        )
    return "\n".join(lines)


FUNDAMENTALS_SCHEMA = {
    "type": "function",
    "function": {
        "name": "get_fundamentals",
        "description": (
            "Look up current headline financials for a stock ticker: price, trailing "
            "and forward P/E, market cap, return on equity, profit margin, revenue "
            "and free cash flow. Use this whenever you need a figure you intend to "
            "reason or compute with, instead of recalling one. "
            "Pass the ticker exactly as given to you and do not substitute a related "
            "one: GOOGL and GOOG are different share classes of Alphabet, with "
            "different prices and different voting rights, and they are not "
            "interchangeable."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "ticker": {
                    "type": "string",
                    "description": "The exact ticker symbol, e.g. 'GOOGL'",
                }
            },
            "required": ["ticker"],
        },
    },
}

In [ ]:
# Stage 4: real financial data. Same loop, one more function.
STAGE_4_SCHEMAS = [CALCULATOR_SCHEMA, WEB_SEARCH_SCHEMA, FUNDAMENTALS_SCHEMA]
STAGE_4_REGISTRY = {
    "calculator": calculator,
    "web_search": web_search,
    "get_fundamentals": get_fundamentals,
}

result = run_agent(client, MODEL, COMPANY, STAGE_4_SCHEMAS, STAGE_4_REGISTRY)
print(result.answer)
print(f"\n--- {result.model_calls} model calls, tools used: {result.tool_calls}")
if result.stopped_early:
    print("--- NOTE: hit the round limit, so the answer above is incomplete.")

# Pick one figure out of that answer and trace it back: which tool produced it,
# and does it match what you find yourself?

## The calculator, revisited

Go back and look at stage 2 again.

Adding a calculator barely helped, and that surprises most people. The arithmetic was
never the problem -- the model can multiply. The problem was that it was multiplying
numbers it had invented. A calculator applied to a made-up price gives you a precisely
wrong answer, which is worse than an obviously wrong one, because it looks like work.

The calculator only became worth having at stage 4, once `get_fundamentals` started
feeding it figures that were actually true. **A tool is only as good as what feeds it.**

That is the thing worth taking out of today. When someone shows you an agent, the
interesting question is not how many tools it has. It is where its facts come from, and
whether you can check them.

A few things we skipped, which matter in practice:

- **Cost.** Each stage printed its model-call count. Stage 4 costs several calls per
  question, not one -- a loop that calls tools is not a single API call.
- **Trust.** `web_search` returns whatever is on the internet, including things that are
  wrong. The agent has no way to tell a filing from a forum post.
- **Not advice.** None of this is investment advice, and a model with a data feed is
  still not a CFA.

In [ ]:
# ---------------------------------------------------------------------------
# Your turn.
#
# 1. Rewrite the prompt. Replace SYSTEM_PROMPT with the analyst you actually
#    want -- a credit analyst, a short seller, an ESG screener -- re-run
#    stage 4, and see what changes.
#
# 2. Add one tool. Fill in the three blanks below. You do not need to touch
#    run_agent: that is the point.
# ---------------------------------------------------------------------------


def my_tool(argument: str) -> str:
    """Say what your tool does -- the model reads this docstring's job in the
    schema below, not here."""
    # Return a STRING. Whatever you return is handed back to the model as text.
    return "..."


MY_TOOL_SCHEMA = {
    "type": "function",
    "function": {
        "name": "my_tool",  # must match the function name in the registry below
        "description": (
            "Tell the model what this does and, just as importantly, when to "
            "reach for it instead of another tool."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "argument": {
                    "type": "string",
                    "description": "What this argument is",
                }
            },
            "required": ["argument"],
        },
    },
}


MY_SCHEMAS = STAGE_4_SCHEMAS + [MY_TOOL_SCHEMA]
MY_REGISTRY = {**STAGE_4_REGISTRY, "my_tool": my_tool}

result = run_agent(client, MODEL, COMPANY, MY_SCHEMAS, MY_REGISTRY)
print(result.answer)
print(f"\n--- {result.model_calls} model calls, tools used: {result.tool_calls}")
if result.stopped_early:
    print("--- NOTE: hit the round limit, so the answer above is incomplete.")